In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

import metpy as metpy
import metpy.calc
from metpy.units import units

In [ ]:
######## Parameter ########

Pa = 101325       # atmosphere pressure Pa
P0 = Pa + 400000      # pressure inside the rocket (Pa)
V0 = 0.0015     # initial air volume inside the rocket m^3



relativehumidity = 1.0
temp = 25 * units.degC
pressure = Pa * units.Pa

mixingratio = metpy.calc.mixing_ratio_from_relative_humidity(pressure, temp, relativehumidity, phase='auto')
density = metpy.calc.density(pressure, temp, mixingratio)

rhoa = density.magnitude # air density (kg/m3)

In [ ]:
######## Parameters - 2 ########

k = 1.4           # heat capacity ratio
rho = 1000.0      # water density kg/m3
VB = 0.002     # bottle volume
A = (0.011**2)*math.pi     # area of the nozzle m^2
Mr = 0.181        # mass of the empty rocket (in kg)
#Ac = 0.007648398
Ac = (0.05**2)*math.pi  # crossectional area of a bottle (m^2)
g = 9.81          # gravitational constant
#Cd = 0.497405         # drag coefficient of the rocket
Cd = 0.497405
C = P0 * (V0**k)
#rhoa = 1.2694842991230404    


t0 = 0.0          
t_end = 10
N = 100000

In [ ]:
t_values = np.linspace(t0, t_end, N+1)
u_values = np.zeros_like(t_values)
m_values = np.zeros_like(t_values)
a_values = np.zeros_like(t_values)
v_values = np.zeros_like(t_values)
h_values = np.zeros_like(t_values)
ft_values = np.zeros_like(t_values)



h = (t_end - t0)/N

In [ ]:
def f(u): 
    somevariable = ((2*Pa + rho*(u**2))/(2*C))
    return (somevariable**((k+1)/k))*((-C * k * A)/rho)


# Yn is in our case u (u(t))
def RK4u(yn,h):
    k1 = f(yn)
    k2 = f(yn + h * k1 / 2)
    k3 = f(yn + h * k2 / 2)
    k4 = f(yn + h * k3)

    return yn + (h / 6)*(k1 + 2*k2 + 2*k3 +k4) 
    # should return Yn+1 or in our case u(t+h)

In [ ]:
def EuM(n): #returns m_values[n+1]
    if m_values[n] > Mr:
        return m_values[n]-h*u_values[n]*A*rho
    else:
        return Mr

In [ ]:
m_values[0]=Mr+(VB-V0)*rho

u0 = np.sqrt(2*(P0-Pa)/rho) # calculating the initial u and m
u_values[0] = u0

In [ ]:
for i in range(N):
    if m_values[i]>Mr:
        u_values[i+1] = RK4u(u_values[i],h)
        m_values[i+1] = EuM(i)
    else: # when the rocket has ran out of water, the mass and u remain constant 
        u_values[i+1] = 0
        m_values[i+1] = Mr

In [ ]:
plt.plot(t_values, u_values)
plt.xlabel('t (s)')
plt.ylabel('u (m/s)')
plt.title('Modelled velocity of exiting water')
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
plt.plot(t_values, m_values, label='m(t)')
plt.xlabel('t (s)')
plt.ylabel('m (kg)')
plt.title('Modelled mass of the rocket')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
def T(n): # retunns Thrust for the step n
    if m_values[n] > Mr:  # if there is no water, there is no thrust
        return A*rho*(u_values[n]**2)
    else:
        return 0

for k in range(N):
    ft_values[k]=T(k)

In [ ]:
plt.figure()
plt.plot(t_values, ft_values, label='T(t)')
plt.xlabel('t (s)')
plt.ylabel('T (N)')
plt.title('Modelled Thrust of the rocket')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
def a(n,v):
    if v_values[n] >= 0: # so that the air drag is always directed against the movement of the rocket and not just downwards 
        return (T(n) - (1/2)*Cd*rhoa*Ac*(v**2) - m_values[n]*g)/m_values[n]
    
    else:
        return (T(n) + (1/2)*Cd*rhoa*Ac*(v**2) - m_values[n]*g)/m_values[n]


def RK1_V(n):   #returns v_values[n+1]
    return v_values[n] + h*a(n,v_values[n])

In [ ]:
for i in range(N):
    v_values[i+1] = RK1_V(i)
    a_values[i] = a(i, v_values[i])

In [ ]:
plt.figure()
plt.plot(t_values, a_values, label='a(t)')
plt.xlabel('t (s)')
plt.ylabel('a (m/s2)')
plt.title('Modelled acceleration of the rocket')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(t_values, v_values, label='v(t)')
plt.xlabel('t (s)')
plt.ylabel('v (m/s)')
plt.title('Modelled velocity of the rocket')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
for b in range(0, N):
    h_values[b] = np.sum(v_values[0:b])*h
    if h_values[b]<0:
        h_values[b]=0
        

In [ ]:
plt.figure()
plt.plot(t_values, h_values, label='h(t)')
plt.xlabel('t (s)')
plt.ylabel('h (m)')
plt.title('Modelled altitude of the rocket')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
print("Max altitude: ", np.max(h_values))